# RSNA Knee Abnormality Detection - Inference (Simplified MedDINOv3)

Using simplified MedDINOv3 backbone with automatic patch handling

In [11]:
# !pip install pydicom timm albumentations

import os, cv2, math, random, warnings
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import timm
from tqdm import tqdm
import multiprocessing as mp
try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

from model_meddinov3_simplified import KneeAbnormalityModelWithMedDINOv3, pad_image

warnings.filterwarnings('ignore')
print('✓ All imports successful')

✓ All imports successful


In [12]:
class InfConfig:
    KAGGLE_DIR      = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Kaggle/rsna-knee-abnormality-detection')
    PYTHON_DIR      = Path('/run/media/mahyar/Samsung-990-PRO/Projects/Python/RSNA Knee Abnormality Detection')
    TEST_SERIES_CSV = KAGGLE_DIR / 'test_series.csv'
    TEST_IMG_DIR    = KAGGLE_DIR / 'test_series'
    CHECKPOINT      = str(PYTHON_DIR / 'best_model_meddinov3.pth')
    SUBMISSION_PATH = str(KAGGLE_DIR / 'submission.csv')

    # Image processing (padded to 14x multiples for MedDINOv3)
    PATCH_SIZE_DINO = 14  # MedDINOv3 uses 14x14 patches internally
    MAX_SLICES      = 12
    LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',
              'Medial OA','Lateral OA','PF OA','Effusion',
              'Synovitis',"Baker's",'Contusion','Fracture']
    NUM_LABELS   = 12
    PLANE_TO_IDX = {'Sagittal': 0, 'Axial': 1, 'Coronal': 2}
    NUM_PLANES   = 3

    # Model dimensions
    SLICE_DIM    = 512
    META_DIM     = 256
    SLICE_ENC_HEADS  = 8
    SLICE_ENC_LAYERS = 2
    DROPOUT      = 0.3

    BATCH_SIZE   = 4
    NUM_WORKERS  = 4
    USE_TTA      = True
    DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

icfg = InfConfig()
print(f'Device: {icfg.DEVICE}')
print(f'MedDINOv3 patch size: {icfg.PATCH_SIZE_DINO}x{icfg.PATCH_SIZE_DINO}')

Device: cuda
MedDINOv3 patch size: 14x14


In [13]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)

In [14]:
# ── Test Dataset ──────────────────────────────────────────────────────────────
class KneeTestDataset(Dataset):
    '''
    Dataset for test inference.
    Images are padded to nearest multiple of 14 (MedDINOv3 patch size).
    '''
    def __init__(self, df_series: pd.DataFrame, img_dir: Path, cfg):
        self.cfg = cfg
        self.img_dir = img_dir
        
        # Group by StudyInstanceUID to collect all series per study
        self.studies = []
        for study_id, group in df_series.groupby('StudyInstanceUID'):
            series_list = []
            for _, row in group.iterrows():
                series_list.append({
                    'series_id': row['SeriesInstanceUID'],
                    'plane': row['Anatomical_Plane'],
                    'plane_id': cfg.PLANE_TO_IDX.get(row['Anatomical_Plane'], 0),
                    'fluid_sensitive': int(row.get('fluid_sensitive', 0)),
                    'fat_suppression': int(row.get('fat_suppression', 0))
                })
            self.studies.append({'study_id': study_id, 'series': series_list})

    def _load_series(self, series_id: str) -> torch.Tensor:
        '''
        Load and normalize DICOM series to (S, 3, H, W).
        '''
        series_dir = self.img_dir / series_id
        dcm_files = sorted(series_dir.glob('*.dcm'))
        
        images = []
        for dcm_file in dcm_files[:self.cfg.MAX_SLICES]:  # Limit to MAX_SLICES
            dcm = pydicom.dcmread(dcm_file)
            img = dcm.pixel_array.astype(np.float32)
            
            # Normalize
            img = (img - img.min()) / (img.max() - img.min() + 1e-6)
            img = cv2.resize(img, (224, 224))
            
            # Convert to 3-channel
            img = np.stack([img, img, img], axis=0)  # (3, 224, 224)
            images.append(img)
        
        # Pad slices to MAX_SLICES
        while len(images) < self.cfg.MAX_SLICES:
            images.append(np.zeros_like(images[0]))
        
        return torch.from_numpy(np.stack(images[:self.cfg.MAX_SLICES]))  # (S, 3, 224, 224)

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, idx):
        study = self.studies[idx]
        study_id = study['study_id']
        
        # Load all series for this study
        all_images = []
        all_masks = []
        all_plane_ids = []
        all_fluid = []
        all_fat = []
        
        for series_info in study['series']:
            images = self._load_series(series_info['series_id'])
            all_images.append(images)  # (S, 3, 224, 224)
            all_plane_ids.append(series_info['plane_id'])
            all_fluid.append(series_info['fluid_sensitive'])
            all_fat.append(series_info['fat_suppression'])
        
        return {
            'study_id': study_id,
            'images_list': all_images,  # List of (S, 3, 224, 224)
            'plane_ids': torch.tensor(all_plane_ids, dtype=torch.long),
            'fluid_sensitive': torch.tensor(all_fluid, dtype=torch.long),
            'fat_suppression': torch.tensor(all_fat, dtype=torch.long),
        }

In [15]:
# ── Collate function ──────────────────────────────────────────────────────────
def collate_fn(batch: List[Dict]) -> Dict:
    '''
    Collate batch of variable-length series.
    Pads to maximum number of series in batch.
    '''
    study_ids = [item['study_id'] for item in batch]
    images_list = []
    plane_ids_list = []
    fluid_list = []
    fat_list = []
    
    max_series = max(len(item['images_list']) for item in batch)
    
    for item in batch:
        images_list.extend(item['images_list'])
        plane_ids_list.extend(item['plane_ids'])
        fluid_list.extend(item['fluid_sensitive'])
        fat_list.extend(item['fat_suppression'])
        
        # Pad series if needed
        num_to_pad = max_series - len(item['images_list'])
        if num_to_pad > 0:
            dummy_img = torch.zeros_like(item['images_list'][0])
            images_list.extend([dummy_img] * num_to_pad)
            plane_ids_list.extend([0] * num_to_pad)  # Dummy plane
            fluid_list.extend([0] * num_to_pad)
            fat_list.extend([0] * num_to_pad)
    
    # Stack: (B*num_series, S, 3, 224, 224)
    images = torch.stack(images_list)
    
    # Pad to multiple of 14 (MedDINOv3 patch size)
    images = pad_image(images, patch_size=14)
    
    return {
        'study_ids': study_ids,
        'images': images,  # (B*num_series, S, 3, H, W)
        'plane_ids': torch.tensor(plane_ids_list, dtype=torch.long),
        'fluid_sensitive': torch.tensor(fluid_list, dtype=torch.long),
        'fat_suppression': torch.tensor(fat_list, dtype=torch.long),
        'num_series': max_series,
    }

In [16]:
# ── Load Model ────────────────────────────────────────────────────────────────
print('Loading model...')
model = KneeAbnormalityModelWithMedDINOv3(icfg).to(icfg.DEVICE)

if os.path.exists(icfg.CHECKPOINT):
    ckpt = torch.load(icfg.CHECKPOINT, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    model.eval()
    print(f"✓ Checkpoint loaded: epoch={ckpt.get('epoch','?')}  "
          f"val_auc={ckpt.get('val_auc', float('nan')):.4f}")
else:
    model.eval()
    print(f"⚠ Checkpoint not found: {icfg.CHECKPOINT}")

print(f"✓ Model on device: {icfg.DEVICE}")

Loading model...
✓ MedDINOv3 loaded (frozen)
  Missing keys: 15
  Unexpected keys: 27
⚠ Checkpoint not found: /run/media/mahyar/Samsung-990-PRO/Projects/Python/RSNA Knee Abnormality Detection/best_model_meddinov3.pth
✓ Model on device: cuda


In [17]:
@torch.no_grad()
def run_inference(model, loader, device, num_series):
    model.eval()
    study_ids, all_probs = [], []
    
    for batch in tqdm(loader, desc='Inference'):
        images = batch['images'].to(device)  # (B*num_series, S, 3, H, W)
        plane_ids = batch['plane_ids'].to(device)
        fluid_sensitive = batch['fluid_sensitive'].to(device)
        fat_suppression = batch['fat_suppression'].to(device)
        
        # Create slice mask (no padding slices in test)
        slice_mask = torch.zeros(images.shape[0], images.shape[1], dtype=torch.bool, device=device)
        
        logits = model(images, slice_mask, plane_ids, fluid_sensitive, fat_suppression)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        study_ids.extend(batch['study_ids'])
    
    return study_ids, np.vstack(all_probs)

print('✓ Inference function ready')

✓ Inference function ready


In [18]:
# ── Run Inference ─────────────────────────────────────────────────────────────
df_test_series = pd.read_csv(icfg.TEST_SERIES_CSV)
print(f'Test series: {len(df_test_series)}')
print(f'Test studies: {df_test_series["StudyInstanceUID"].nunique()}')
print('Planes:', df_test_series['Anatomical_Plane'].value_counts().to_dict())

ds = KneeTestDataset(df_test_series, icfg.TEST_IMG_DIR, icfg)
ld = DataLoader(ds, batch_size=icfg.BATCH_SIZE, shuffle=False,
                num_workers=icfg.NUM_WORKERS, pin_memory=True,
                collate_fn=collate_fn)

study_ids, probs = run_inference(model, ld, icfg.DEVICE, 1)
print(f'\\nGenerated predictions for {len(study_ids)} studies')
print(f'Probabilities shape: {probs.shape}')
print(f'Probability range: [{probs.min():.4f}, {probs.max():.4f}]')

Test series: 15
Test studies: 3
Planes: {'Sagittal': 7, 'Axial': 4, 'Coronal': 4}


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


IndexError: Caught IndexError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/run/media/mahyar/Samsung-990-PRO/Projects/Python/RSNA Knee Abnormality Detection/.venv/lib64/python3.14/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/run/media/mahyar/Samsung-990-PRO/Projects/Python/RSNA Knee Abnormality Detection/.venv/lib64/python3.14/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_4152924/3313308728.py", line 66, in __getitem__
    images = self._load_series(series_info['series_id'])
  File "/tmp/ipykernel_4152924/3313308728.py", line 47, in _load_series
    images.append(np.zeros_like(images[0]))
                                ~~~~~~^^^
IndexError: list index out of range


In [ ]:
# ── Create Submission ─────────────────────────────────────────────────────────
submission = pd.DataFrame()
submission['study_id'] = study_ids
for i, label in enumerate(icfg.LABELS):
    submission[label] = probs[:, i]

submission.to_csv(icfg.SUBMISSION_PATH, index=False)
print(f'✓ Submission saved to {icfg.SUBMISSION_PATH}')
print(submission.head())